In [1]:
import time
import pandas as pd
import requests
from tqdm.auto import tqdm

NETID = "bdowlin2"
PROJECTS = pd.read_csv("net2prj.csv").query("netID == @NETID").rename(
    columns={"WoC": "project_wocid", "GH": "github_url"}
)
PROJECTS

C:\Users\brett\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,netID,project_wocid,github_url
460,bdowlin2,avehtari_bda_course_aalto,https://github.com/avehtari/BDA_course_Aalto
461,bdowlin2,eberharf_cfl,https://github.com/eberharf/cfl
462,bdowlin2,openda-association_openda,https://github.com/OpenDA-Association/OpenDA
463,bdowlin2,openms_openms,https://github.com/OpenMS/OpenMS
464,bdowlin2,vatlab_sos,https://github.com/vatlab/SoS
465,bdowlin2,jayggg_ngstents,https://github.com/jayggg/ngstents
466,bdowlin2,inqwire_sqir,https://github.com/inQWIRE/SQIR
467,bdowlin2,tee-lab_pyddsde,https://github.com/tee-lab/pyddsde
468,bdowlin2,jgcri_gcam-core,https://github.com/JGCRI/gcam-core
469,bdowlin2,windweller_disextract,https://github.com/windweller/DisExtract


## Execution environment

Run this notebook on Hydra or Google Colab. Before running the WoC cell, install the API client with `%pip install -U python-woc pandas requests tqdm` and configure a WoC API key if your account requires one.

In [2]:
from woc.remote import WocMapsRemote

# Use an API key here when Hydra/Colab provides one.
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

project_commits = []
for project in PROJECTS.itertuples(index=False):
    shas = woc.get_values("p2c", project.project_wocid)
    project_commits.extend(
        {"project_wocid": project.project_wocid, "commit_sha1": sha}
        for sha in shas
    )

commits = pd.DataFrame(project_commits).drop_duplicates()
commits.to_csv(f"{NETID}_commit_shas.csv", index=False)
print(f"Retrieved {len(commits):,} project-commit associations across {len(PROJECTS)} projects.")
commits.head()

ModuleNotFoundError: No module named 'woc'

In [ ]:
commit_rows = []
sha_batches = [
    commits.iloc[start:start + 10]
    for start in range(0, len(commits), 10)
]

for batch in tqdm(sha_batches, desc="Retrieving WoC commit data"):
    response, errors = woc.get_values_many("commit.tch", batch["commit_sha1"].tolist())
    if errors:
        print("WoC errors:", errors)
    for sha1, values in response.items():
        commit = values[0]
        project_wocids = batch.loc[batch["commit_sha1"] == sha1, "project_wocid"]
        for project_wocid in project_wocids:
            commit_rows.append({
                "project_wocid": project_wocid,
                "commit_sha1": sha1,
                "author": commit[2][0],
                "time": int(commit[2][1]),
                "commit message": commit[4],
            })
    time.sleep(1)

summary = pd.DataFrame(commit_rows).drop_duplicates(
    subset=["project_wocid", "commit_sha1"]
)
summary.head()

In [ ]:
# WoC statistics required for each project.
woc_stats = (
    summary.groupby("project_wocid")
    .agg(
        number_of_commits=("commit_sha1", "nunique"),
        number_of_authors=("author", "nunique"),
        max_time=("time", "max"),
        min_time=("time", "min"),
    )
    .reset_index()
)
woc_stats

In [ ]:
def github_metadata(github_url):
    owner, repo = github_url.rstrip("/").split("github.com/")[1].split("/", 1)
    api_url = f"https://api.github.com/repos/{owner}/{repo}"
    repository = requests.get(api_url, timeout=30)
    repository.raise_for_status()
    details = repository.json()

    recent_commit = requests.get(
        f"{api_url}/commits", params={"per_page": 1}, timeout=30
    )
    recent_commit.raise_for_status()
    commit_data = recent_commit.json()
    last_commit_date = None
    if commit_data:
        last_commit_date = commit_data[0]["commit"]["committer"]["date"]
    return {
        "number_of_stars": details["stargazers_count"],
        "number_of_forks": details["forks_count"],
        "last_commit_date": last_commit_date,
    }

github_rows = []
for project in PROJECTS.itertuples(index=False):
    metadata = github_metadata(project.github_url)
    github_rows.append({"project_wocid": project.project_wocid, **metadata})
    print(
        f"{project.project_wocid}: stars={metadata['number_of_stars']}, "
        f"forks={metadata['number_of_forks']}, "
        f"last commit={metadata['last_commit_date']}"
    )

github_stats = pd.DataFrame(github_rows)
stats = woc_stats.merge(github_stats, on="project_wocid")
stats

In [ ]:
summary.to_csv(
    f"{NETID}_project_summary.csv",
    columns=["project_wocid", "commit_sha1", "author", "time", "commit message"],
    index=False,
    sep=";",
)
print(f"Wrote {len(summary):,} rows to {NETID}_project_summary.csv")
summary.head()